# 09.2 - Autonomous Literature Review Agent with AIMU Workflows

This notebook composes an autonomous literature review agent using AIMU's agent workflow primitives ([`Chain`](https://github.com/InstituteforDiseaseModeling/aimu), `EvaluatorOptimizer`, `Agent`).

Pipeline:

1. **Searcher** - an `Agent` with access to the genscai MCP `search_research_articles` tool. It autonomously decides what queries to issue and how many searches to run against the local medRxiv vector database.
2. **Analyst** - an `Agent` that extracts methods, datasets, findings, and gaps from the retrieved literature.
3. **Writer + Critic** - a writer agent wrapped in an `EvaluatorOptimizer`, which iteratively revises the synthesized review until a critic agent accepts it.

These three steps are composed into a single `Chain`. The text output of each step becomes the task input of the next.

## 01. Setup

Verify the medRxiv vector database is present. If not, run notebooks 01 and 03 (or scripts 99.1) to download articles and build the embeddings.

In [ ]:
from genscai import paths
import chromadb

DB_PATH = str(paths.output / "medrxiv.db")

client = chromadb.PersistentClient(path=DB_PATH)
client.list_collections()

Wire up the genscai MCP server so the searcher agent can call `search_research_articles`. `nest_asyncio` is required for MCP clients to run inside Jupyter's event loop.

In [ ]:
from aimu.tools import MCPClient

import nest_asyncio

nest_asyncio.apply()

mcp_client = MCPClient()

for tool in mcp_client.list_tools():
    print(f"{tool.name}: {tool.description}")

## 02. Model Client

All agents in this notebook share one underlying model. AIMU's `Chain.from_config` wires a single `BaseModelClient` across all steps and resets `messages` between steps so each agent runs in isolation. Pick a model that supports tool calling (the searcher needs MCP tool access).

In [ ]:
from aimu.models import OllamaClient, OllamaModel

model_client = OllamaClient(OllamaModel.QWEN_3_5_9B)
model_client.mcp_client = mcp_client

## 03. Define the Workflow Stages

Each stage is an `Agent` config: a system message that defines the role and a `max_iterations` cap on the agentic tool-use loop. Only the searcher needs many iterations (it autonomously decides how many tool calls to make); the downstream agents work over text already gathered.

In [ ]:
SEARCHER_SYSTEM = (
    "You are a literature search agent. Given a research topic, autonomously plan and execute "
    "multiple complementary searches against the `search_research_articles` tool to build broad coverage "
    "of the topic. Vary your queries to capture different methods, populations, and time periods. "
    "After you have enough material (at least 3-4 searches), stop calling tools and return the raw "
    "article titles, abstracts, authors, and links you collected, grouped by sub-topic."
)

ANALYST_SYSTEM = (
    "You are a research analyst. Given a corpus of article abstracts, extract a structured analysis "
    "with these sections: (1) Methods used, (2) Datasets and populations, (3) Key findings, "
    "(4) Open questions and gaps. Cite article titles inline. Be concise, factual, and avoid "
    "speculation beyond what the abstracts state."
)

WRITER_SYSTEM = (
    "You are a scientific writer. Turn the structured analysis into a coherent literature review "
    "of 4-6 paragraphs. Open with the scope and motivation, organize the body by theme (not by paper), "
    "weave inline citations using article titles, and close with a synthesis of gaps and future work."
)

CRITIC_SYSTEM = (
    "You are a peer reviewer. Evaluate the literature review for: thematic organization (not "
    "paper-by-paper summary), accurate inline citations, balanced coverage, and a gaps/future-work "
    "synthesis. If the review meets all four criteria, reply with exactly PASS. Otherwise reply "
    "REVISE: <specific, actionable feedback>."
)

## 04. Compose the Workflow

Build the searcher and analyst as `Agent`s, then wrap the writer in an `EvaluatorOptimizer` so the critic can request revisions. Finally chain all three stages with `Chain`.

We construct the agents directly (rather than `Chain.from_config`) because the writer step is itself a workflow, not a plain agent.

In [ ]:
from aimu.agents import Agent, Chain, EvaluatorOptimizer

searcher = Agent(
    model_client=model_client,
    name="searcher",
    system_message=SEARCHER_SYSTEM,
    max_iterations=8,
    reset_messages_on_run=True,
)

analyst = Agent(
    model_client=model_client,
    name="analyst",
    system_message=ANALYST_SYSTEM,
    max_iterations=2,
    reset_messages_on_run=True,
)

writer = Agent(
    model_client=model_client,
    name="writer",
    system_message=WRITER_SYSTEM,
    max_iterations=2,
    reset_messages_on_run=True,
)

critic = Agent(
    model_client=model_client,
    name="critic",
    system_message=CRITIC_SYSTEM,
    max_iterations=2,
    reset_messages_on_run=True,
)

writer_loop = EvaluatorOptimizer(
    generator=writer,
    evaluator=critic,
    max_rounds=3,
    pass_keyword="PASS",
)

review_pipeline = Chain(agents=[searcher, analyst, writer_loop])

## 05. Run the Pipeline

Stream the chain so each stage's output appears as it's produced. `ChainChunk` carries the step index and the agent name, which is useful for tracing what each stage emits and when the writer/critic loop revises.

In [ ]:
from aimu.models import StreamPhase

TOPIC = "agent-based models for malaria transmission in sub-Saharan Africa"

current_step = -1
for chunk in review_pipeline.run(TOPIC, stream=True):
    if chunk.step != current_step:
        current_step = chunk.step
        print(f"\n\n========== step {chunk.step}: {chunk.agent_name} ==========\n", flush=True)
    if chunk.phase == StreamPhase.TOOL_CALLING:
        print(f"\n[tool: {chunk.content['name']}]\n", flush=True)
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## 06. Inspect the Final Review

Run the chain non-streaming to capture the final literature review as a single string.

In [ ]:
review = review_pipeline.run(TOPIC)
print(review)

Inspect the per-agent message histories. Because all four agents share one `model_client`, every key in the dict points at the most recently used messages list (the critic's last turn). To preserve full per-stage history, give each agent its own `ModelClient` instance.

In [ ]:
list(review_pipeline.messages.keys())

## 07. Cleanup

Drop the model client to free GPU memory before running another notebook against the same device.

In [ ]:
del model_client